# Modelo de Traslado de Valores - FINABIEN
## 03. Entrenamiento de Modelos
### El propósito de este 'notebook' es generar los conjutnos de datos de entrenamiento/validación, entrenar modelos específicos para series de tiempo y elegir el modelo con la mejor efetividad en el pronóstico del flujo de efectivo de la sucursal. 

In [2]:
import joblib
import os
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import holidays
from pandas.tseries.offsets import CustomBusinessDay
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

## 3.1 Estructura de los datos
### Cargamos la base de datos y analizamos la estructura de los datos

In [ ]:
# Ruta donde se guardó el archivo de los días festivos
ruta_archivo_flujo_efectivo_estado = "C:/Users/jose.luis.flores/Documents/Repositorios/MTDV/2. Datos/flujo_estado.pkl"

with open(ruta_archivo_flujo_efectivo_estado, "rb") as archivo:
    flujo_estado = pickle.load(archivo)

print("Base de datos cargada correctamente.")

In [5]:
# Seleccionar solo las columnas deseadas
columnas_deseadas = [
    'fecha', 'nombre_estado', 'id_sucursal', 'entradas', 'salidas','flujo_efectivo'
]
flujo_estado = flujo_estado[columnas_deseadas]

In [ ]:
flujo_estado.head()

## 3.2 Creación de características temporales
### Generamos las características necesarias para el entrenamiento de los modelos de series de tiempo

In [ ]:
# Verificar que no hay Sábados (5) o Domingos (6)
dias_semana = flujo_estado['fecha'].dt.dayofweek  # Lunes=0, Domingo=6
print("Días de la semana presentes:", dias_semana.unique())  # Debe mostrar [0, 1, 2, 3, 4]

In [ ]:
# 1. Definir días festivos MX
festivos_mx = holidays.MX(years=flujo_estado['fecha'].dt.year.unique())

# 2. Filtrar solo días laborales (excluyendo Sáb/Dom y festivos)
flujo_estado = flujo_estado[
    (flujo_estado['fecha'].dt.dayofweek < 5) &  # L-V (0-4)
    (~flujo_estado['fecha'].isin(festivos_mx))  # Excluye festivos
].copy()

In [10]:
# Día de la semana (0=Lunes, 4=Viernes)
flujo_estado['dia_semana'] = flujo_estado['fecha'].dt.dayofweek

# Semana del año
flujo_estado['semana_año'] = flujo_estado['fecha'].dt.isocalendar().week

# Mes 
flujo_estado['mes'] = flujo_estado['fecha'].dt.month

#Año
flujo_estado['año'] = flujo_estado['fecha'].dt.year

In [11]:
# Media móvil de 5 y 10 días laborales
flujo_estado['media_movil_5'] = flujo_estado['flujo_efectivo'].rolling(5).mean()
flujo_estado['media_movil_10'] = flujo_estado['flujo_efectivo'].rolling(10).mean()

In [12]:
# Rezagos de 1, 2, 3 y 5 días laborales
for lag in [1, 2, 3, 5]:
    flujo_estado[f'lag_{lag}'] = flujo_estado['flujo_efectivo'].shift(lag)

In [ ]:
flujo_estado.head(10)

In [ ]:
# Eliminar filas con al menos un NaN en cualquier columna
flujo_estado = flujo_estado.dropna(how='any')

# Verificar resultado
print(f"Filas después de eliminar NaN: {len(flujo_estado)}")

In [ ]:
flujo_estado.head()

In [16]:
# Ruta donde se guardará el archivo
ruta_archivo_flujo_estado_características = "C:/Users/jose.luis.flores/Documents/Repositorios/MTDV/2. Datos/flujo_estado_características.pkl"

# Guardar el DataFrame en un archivo pickle
with open(ruta_archivo_flujo_estado_características, "wb") as archivo:
    pickle.dump(flujo_estado, archivo)

print(f"Base de datos guardada en {ruta_archivo_flujo_estado_características}")

Base de datos guardada en C:/Users/jose.luis.flores/Documents/Repositorios/MTDV/2. Datos/flujo_estado_características.pkl


In [ ]:
flujo_estado.dtypes

## 3.3 Separación de los conjuntos de Entrenamiento y Validación
### Hacemos la separación de los datos en Entrenamiento/Validación para entenar los modelos y conocer las métricas de error.

In [ ]:
# Número de días para validación
n_validacion = 10

# Crear diccionarios para guardar los datos separados por sucursal
entrenamientos = {}
validaciones = {}

# Agrupar por sucursal
for id_sucursal, df_sucursal in flujo_estado.groupby('id_sucursal'):
    # Ordenar por fecha (por si acaso)
    df_sucursal = df_sucursal.sort_values('fecha')

    # Definir el índice de corte para validación
    split_idx = len(df_sucursal) - n_validacion

    # Separar entrenamiento y validación
    entrenamiento = df_sucursal.iloc[:split_idx].copy()
    validacion = df_sucursal.iloc[split_idx:].copy()

    # Guardar en los diccionarios
    entrenamientos[id_sucursal] = entrenamiento
    validaciones[id_sucursal] = validacion

    # Verificación por sucursal
    print(f"Sucursal {id_sucursal}:")
    print(f"  Entrenamiento: {entrenamiento['fecha'].min()} a {entrenamiento['fecha'].max()} ({len(entrenamiento)} filas)")
    print(f"  Validación: {validacion['fecha'].min()} a {validacion['fecha'].max()} ({len(validacion)} filas)")
    print("-" * 50)

In [ ]:
# Crear diccionario para guardar los datos filtrados por días laborales
entrenamientos_filtrados = {}

# Días de la semana (0=Lunes, 6=Domingo)
dias_semana_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

# Iterar por cada sucursal
for id_sucursal, entrenamiento in entrenamientos.items():
    # Asegurar que la fecha esté en datetime
    entrenamiento['fecha'] = pd.to_datetime(entrenamiento['fecha'])

    # Identificar días únicos en que opera esta sucursal
    dias_laborales_sucursal = sorted(entrenamiento['fecha'].dt.dayofweek.unique())

    # Mapear los días al formato que espera weekmask
    weekmask = ' '.join([dias_semana_labels[d] for d in dias_laborales_sucursal])

    # Crear frecuencia personalizada para esta sucursal
    frecuencia_sucursal = CustomBusinessDay(weekmask=weekmask)

    # Crear rango de fechas laborales personalizado
    fecha_min = entrenamiento['fecha'].min()
    fecha_max = entrenamiento['fecha'].max()
    fechas_laborales = pd.date_range(start=fecha_min, end=fecha_max, freq=frecuencia_sucursal)

    # Filtrar el DataFrame original para conservar solo los días laborales reales
    entrenamiento_filtrado = entrenamiento[entrenamiento['fecha'].isin(fechas_laborales)].copy()

    # Guardar en el diccionario
    entrenamientos_filtrados[id_sucursal] = entrenamiento_filtrado

    # Verificación
    print(f"Sucursal {id_sucursal} (días laborales: {weekmask}):")
    print(entrenamiento_filtrado[['fecha', 'flujo_efectivo']].head())
    print("-" * 60)

In [ ]:
# Configurar estilo y tamaño
plt.figure(figsize=(14, 6))
plt.style.use('seaborn-v0_8-darkgrid')

# Graficar entrenamiento (azul)
plt.plot(entrenamiento['fecha'], 
         entrenamiento['flujo_efectivo'], 
         label='Entrenamiento', 
         color='blue', 
         linewidth=1.5)

# Graficar validación (naranja)
plt.plot(validacion['fecha'], 
         validacion['flujo_efectivo'], 
         label='Validación (últimos 10 días)', 
         color='orange', 
         linewidth=2.0,
         linestyle='--')

# Añadir línea vertical para marcar la división
split_date = validacion['fecha'].iloc[0]
plt.axvline(x=split_date, 
            color='red', 
            linestyle=':', 
            linewidth=1.5, 
            alpha=0.7,
            label='Inicio Validación')

# Formatear ejes y leyenda
plt.xlabel('Fecha', fontsize=12)
plt.ylabel('Flujo de Efectivo', fontsize=12)
plt.title('Serie de Tiempo: Entrenamiento vs. Validación', fontsize=14, pad=20)
plt.legend(fontsize=10, loc='upper left')

# Ajustar formato de fechas
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())
plt.gcf().autofmt_xdate()  # Rotar fechas para mejor visualización

# Resaltar puntos de validación
plt.scatter(validacion['fecha'], 
            validacion['flujo_efectivo'], 
            color='red', 
            s=30, 
            zorder=5)

plt.tight_layout()
plt.show()

In [22]:
# Columnas numéricas a incluir (excluyendo 'fecha', 'nombre_estado', 'id_sucursal')
features = ['entradas', 'salidas','dia_semana', 'semana_año', 'mes', 'año', 
            'media_movil_5', 'media_movil_10', 'lag_1', 'lag_2', 'lag_3', 'lag_5']
target = 'flujo_efectivo'

In [ ]:
# Diccionarios para guardar resultados por sucursal
X_trains = {}
X_vals = {}
y_trains = {}
y_vals = {}
scalers_X = {}
scalers_y = {}

# Iterar por sucursal
for id_sucursal in entrenamientos.keys():
    # Datos de entrenamiento y validación
    df_train = entrenamientos[id_sucursal]
    df_val = validaciones[id_sucursal]

    # Escalador de características
    scaler_X = MinMaxScaler()
    X_train = scaler_X.fit_transform(df_train[features])
    X_val = scaler_X.transform(df_val[features])

    # Escalador para la variable objetivo
    scaler_y = MinMaxScaler()
    y_train = scaler_y.fit_transform(df_train[[target]])
    y_val = scaler_y.transform(df_val[[target]])

    # Guardar escaladores y datos transformados
    scalers_X[id_sucursal] = scaler_X
    scalers_y[id_sucursal] = scaler_y
    X_trains[id_sucursal] = X_train
    X_vals[id_sucursal] = X_val
    y_trains[id_sucursal] = y_train
    y_vals[id_sucursal] = y_val

    # Verificación opcional
    print(f"Escalado completado para sucursal {id_sucursal} (X: {X_train.shape}, y: {y_train.shape})")

In [24]:
# Ruta base del proyecto
ruta_base = "C:/Users/jose.luis.flores/Documents/Repositorios/MTDV"

# Modelos considerados
modelos = ["lstm", "xgboost"]

# Crear carpetas necesarias para cada modelo
for modelo in modelos:
    os.makedirs(os.path.join(ruta_base, "escaladores", modelo, "features"), exist_ok=True)
    os.makedirs(os.path.join(ruta_base, "escaladores", modelo, "targets"), exist_ok=True)

# Guardar escaladores por sucursal
for id_sucursal in scalers_X.keys():
    for modelo in modelos:
        # Rutas completas para guardar escaladores
        ruta_scaler_X = os.path.join(ruta_base, "escaladores", modelo, "features", f"sucursal_{id_sucursal}_scaler_X.pkl")
        ruta_scaler_y = os.path.join(ruta_base, "escaladores", modelo, "targets", f"sucursal_{id_sucursal}_scaler_y.pkl")
        
        # Guardar los escaladores
        joblib.dump(scalers_X[id_sucursal], ruta_scaler_X)
        joblib.dump(scalers_y[id_sucursal], ruta_scaler_y)

In [ ]:
# Diccionarios para almacenar los datos reformateados para LSTM
X_trains_lstm = {}
X_vals_lstm = {}

# Reformatear para cada sucursal
for id_sucursal in X_trains.keys():
    X_train = X_trains[id_sucursal]
    X_val = X_vals[id_sucursal]

    # Reformatear a [muestras, pasos de tiempo, características]
    X_train_lstm = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
    X_val_lstm = X_val.reshape((X_val.shape[0], 1, X_val.shape[1]))

    # Guardar en los diccionarios
    X_trains_lstm[id_sucursal] = X_train_lstm
    X_vals_lstm[id_sucursal] = X_val_lstm

    # Verificación opcional
    print(f"Sucursal {id_sucursal}: X_train_lstm shape = {X_train_lstm.shape}, X_val_lstm shape = {X_val_lstm.shape}")

## 3.3 Entrenamiento Incial de los Modelos
### Entrenamos con el conjunto de entrenamiento los modelos LSTM y XGBoost. 

In [27]:
from xgboost import XGBRegressor
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import MeanSquaredError
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:
# Diccionarios para almacenar resultados
modelos_xgb = {}
predicciones_xgb = {}

for id_sucursal in X_trains.keys():
    # Obtener datos escalados
    X_train = X_trains[id_sucursal]
    y_train = y_trains[id_sucursal]
    X_val = X_vals[id_sucursal]
    scaler_y = scalers_y[id_sucursal]

    # Entrenar modelo XGBoost
    modelo_xgb = XGBRegressor(
        objective='reg:squarederror',
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6
    )
    modelo_xgb.fit(X_train, y_train.flatten())

    # Guardar modelo
    modelos_xgb[id_sucursal] = modelo_xgb

    # Hacer predicciones e invertir el escalado
    pred_scaled = modelo_xgb.predict(X_val).reshape(-1, 1)
    pred_invertida = scaler_y.inverse_transform(pred_scaled)

    # Guardar predicciones
    predicciones_xgb[id_sucursal] = pred_invertida

    # Verificación opcional
    print(f"Sucursal {id_sucursal}: predicciones generadas ({len(pred_invertida)} valores)")

In [ ]:
# Diccionarios para guardar modelos y predicciones
modelos_lstm = {}
predicciones_lstm = {}

# Entrenamiento por sucursal
for id_sucursal in X_trains_lstm.keys():
    # Obtener datos
    X_train = X_trains_lstm[id_sucursal]
    y_train = y_trains[id_sucursal]
    X_val = X_vals_lstm[id_sucursal]
    scaler_y = scalers_y[id_sucursal]

    # Definir modelo LSTM
    modelo_lstm = Sequential([
        LSTM(64, activation='relu', input_shape=(X_train.shape[1], X_train.shape[2])),
        Dense(1)
    ])
    modelo_lstm.compile(optimizer=Adam(learning_rate=0.001), loss=MeanSquaredError())

    # Entrenar modelo
    modelo_lstm.fit(X_train, y_train, epochs=50, batch_size=16, verbose=0)

    # Guardar modelo
    modelos_lstm[id_sucursal] = modelo_lstm

    # Predicción y desescalado
    pred_scaled = modelo_lstm.predict(X_val, verbose=0)
    pred_invertida = scaler_y.inverse_transform(pred_scaled)

    # Guardar predicciones
    predicciones_lstm[id_sucursal] = pred_invertida

    # Verificación opcional
    print(f"Sucursal {id_sucursal}: modelo LSTM entrenado y {len(pred_invertida)} predicciones generadas")

In [30]:
def evaluar(y_real, y_pred, nombre_modelo):
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    mae = mean_absolute_error(y_real, y_pred)
    print(f"{nombre_modelo} - RMSE: {rmse:.2f}, MAE: {mae:.2f}")
    return mae

In [ ]:
# Evaluar modelos por sucursal
resultados_mae = []

for id_sucursal in validaciones.keys():
    # Valores reales
    y_real = validaciones[id_sucursal][target].values.reshape(-1, 1)

    # Predicciones
    pred_xgb = predicciones_xgb[id_sucursal]
    pred_lstm = predicciones_lstm[id_sucursal]

    # Evaluación
    print(f"\n📍 Evaluación Sucursal {id_sucursal}")
    mae_xgb = evaluar(y_real, pred_xgb, "XGBoost")
    mae_lstm = evaluar(y_real, pred_lstm, "LSTM")

    # Guardar resultados
    resultados_mae.append({
        'id_sucursal': id_sucursal,
        'mae_xgb': mae_xgb,
        'mae_lstm': mae_lstm
    })

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(validacion['fecha'], validacion[target], label='Real', marker='o')
plt.plot(validacion['fecha'], pred_xgb, label='XGBoost', linestyle='--')
plt.plot(validacion['fecha'], pred_lstm, label='LSTM', linestyle='-.')
plt.title('Predicciones vs Valores Reales (Últimos 10 Días)')
plt.xlabel('Fecha')
plt.ylabel('Flujo de Efectivo')
plt.legend()
plt.grid(True)
plt.show()

## 3.4 Guardar los Modelos Entrenados

In [ ]:
# Ruta base del proyecto
ruta_base = "C:/Users/jose.luis.flores/Documents/Repositorios/MTDV"

# Rutas específicas para guardar modelos
ruta_modelos_lstm = os.path.join(ruta_base, "modelos", "mejores", "lstm")
ruta_modelos_xgb = os.path.join(ruta_base, "modelos", "mejores", "xgboost")

# Crear carpetas si no existen
os.makedirs(ruta_modelos_lstm, exist_ok=True)
os.makedirs(ruta_modelos_xgb, exist_ok=True)

# Diccionario para registrar qué modelo fue mejor por sucursal
mejores_modelos = {}

for id_sucursal in validaciones.keys():
    # Valores reales
    y_real = validaciones[id_sucursal][target].values.reshape(-1, 1)

    # Predicciones
    pred_xgb = predicciones_xgb[id_sucursal]
    pred_lstm = predicciones_lstm[id_sucursal]

    # Evaluación
    print(f"\n📍 Evaluación Sucursal {id_sucursal}")
    mae_xgb = evaluar(y_real, pred_xgb, "XGBoost")
    mae_lstm = evaluar(y_real, pred_lstm, "LSTM")

    # Comparar y guardar el mejor modelo
    if mae_xgb < mae_lstm:
        mejor_modelo = "XGBoost"
        mejor_pred = pred_xgb
        try:
            ruta_modelo = os.path.join(ruta_modelos_xgb, f"sucursal_{id_sucursal}_mejor_modelo_xgb.pkl")
            joblib.dump(modelos_xgb[id_sucursal], ruta_modelo)
            print(f"✅ Modelo guardado: {ruta_modelo} (XGBoost)")
        except Exception as e:
            print(f"❌ Error al guardar modelo XGBoost para sucursal {id_sucursal}: {e}")
    else:
        mejor_modelo = "LSTM"
        mejor_pred = pred_lstm
        try:
            ruta_modelo = os.path.join(ruta_modelos_lstm, f"sucursal_{id_sucursal}_mejor_modelo_lstm.h5")
            # 🔧 Guardar el modelo LSTM sin el optimizador para evitar errores al cargar
            modelos_lstm[id_sucursal].compile(optimizer=Adam(learning_rate=0.001), loss=MeanSquaredError())
            modelos_lstm[id_sucursal].save(ruta_modelo, include_optimizer=False)
            print(f"✅ Modelo guardado: {ruta_modelo} (LSTM)")
        except Exception as e:
            print(f"❌ Error al guardar modelo LSTM para sucursal {id_sucursal}: {e}")

    # Registrar el mejor modelo
    mejores_modelos[id_sucursal] = mejor_modelo

In [ ]:
# Ruta donde guardar el diccionario
ruta_base = "C:/Users/jose.luis.flores/Documents/Repositorios/MTDV"
ruta_pickle = os.path.join(ruta_base, "modelos", "mejores", "mejores_modelos.pkl")

# Guardar el diccionario
joblib.dump(mejores_modelos, ruta_pickle)
print(f"✅ Diccionario 'mejores_modelos' guardado en: {ruta_pickle}")